# SaludAI - Benchmark FHIR-AgentBench

Este notebook permite ejecutar y analizar el benchmark de SaludAI, inspirado en [FHIR-AgentBench](https://arxiv.org/abs/2509.19319) y adaptado para datos clinicos argentinos.

El benchmark evalua 50 preguntas en 3 niveles de complejidad:
- **Simple** (8): Conteos directos, busquedas basicas
- **Medium** (20): Filtros demograficos, condiciones clinicas
- **Complex** (22): Cross-resource, agregaciones, calculos

## Prerequisitos

```bash
docker compose up -d          # HAPI FHIR con datos argentinos
cp .env.example .env          # Configurar API keys
```

## 1. Explorar el dataset

In [ ]:
import sys

sys.stdout.reconfigure(encoding="utf-8")

from collections import Counter

from benchmarks.dataset import load_dataset

questions = load_dataset()

# Resumen del dataset
category_counts = Counter(q.category for q in questions)
subcategory_counts = Counter(q.subcategory for q in questions)

print(f"Total de preguntas: {len(questions)}\n")
print("Por categoria:")
for cat in ["simple", "medium", "complex"]:
    print(f"  {cat}: {category_counts[cat]}")

print(f"\nPor subcategoria:")
for sub, count in subcategory_counts.most_common():
    print(f"  {sub}: {count}")

### Ejemplos de preguntas por categoria

In [ ]:
for cat in ["simple", "medium", "complex"]:
    cat_questions = [q for q in questions if q.category == cat]
    example = cat_questions[0]
    print(f"[{cat.upper()}] {example.id}: {example.question}")
    print(f"  Respuesta esperada: {example.expected_answer}")
    print(f"  Subcategoria: {example.subcategory}")
    if example.notes:
        print(f"  Notas para el judge: {example.notes}")
    print()

## 2. Ejecutar el benchmark

Podemos ejecutar todas las preguntas o filtrar por categoria. Cada pregunta se envia al agente y la respuesta se evalua con un judge hibrido (programatico + LLM).

> **Nota:** Ejecutar las 50 preguntas toma ~30 minutos con Claude Sonnet. Para una prueba rapida, usa una sola categoria.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from benchmarks.config import BenchmarkConfig
from benchmarks.harness import EvalHarness
from benchmarks.judge import AnswerJudge
from benchmarks.metrics import compute_metrics
from benchmarks.results import print_summary
from saludai_agent.config import AgentConfig
from saludai_agent.llm import create_llm_client
from saludai_agent.loop import AgentLoop
from saludai_agent.tracing import create_tracer
from saludai_core.config import FHIRConfig
from saludai_core.fhir_client import FHIRClient
from saludai_core.locales import load_locale_pack
from saludai_core.terminology import TerminologyResolver

# Configurar agente y judge
agent_config = AgentConfig()
bench_config = BenchmarkConfig()

print(f"Agente: {agent_config.llm_provider}/{agent_config.llm_model}")
print(f"Judge: {bench_config.judge_provider}/{bench_config.judge_model}")
print(f"Timeout por pregunta: {bench_config.question_timeout_seconds}s")

In [ ]:
# Ejecutar benchmark (cambiar category_filter para probar un subconjunto)
category_filter = "simple"  # Opciones: "simple", "medium", "complex", o None para todas

filtered = [q for q in questions if category_filter is None or q.category == category_filter]
print(f"Ejecutando {len(filtered)} preguntas (categoria: {category_filter or 'todas'})...\n")

fhir_config = FHIRConfig(fhir_server_url="http://localhost:8080/fhir")
llm = create_llm_client(agent_config)
judge_llm = create_llm_client(AgentConfig(
    llm_provider=bench_config.judge_provider,
    llm_model=bench_config.judge_model,
))

async with FHIRClient(config=fhir_config) as client:
    loop = AgentLoop(
        llm=llm,
        fhir_client=client,
        terminology_resolver=TerminologyResolver(),
        config=agent_config,
        tracer=create_tracer(agent_config),
        locale_pack=load_locale_pack("ar"),
    )
    judge = AnswerJudge(llm=judge_llm, config=bench_config)
    harness = EvalHarness(agent_loop=loop, judge=judge, config=bench_config)
    results = await harness.run_all(filtered)

print(f"\nEvaluacion completada: {len(results)} preguntas procesadas")

## 3. Analizar resultados

In [ ]:
metrics = compute_metrics(results)
print_summary(metrics)

### Detalle por pregunta

In [ ]:
for r in results:
    is_correct = r.correctness_score >= 1.0
    status = "CORRECT" if is_correct else ("ERROR" if r.error else "INCORRECT")
    icon = {"CORRECT": "+", "INCORRECT": "-", "ERROR": "!"}[status]
    dur = f"{r.duration_seconds:.1f}s"
    print(f"  [{icon}] {r.question_id} ({r.category}) — {dur}, {r.iterations} iters")
    if not is_correct:
        print(f"      Pregunta: {r.question}")
        print(f"      Esperado: {r.expected_answer}")
        print(f"      Agente:   {r.agent_answer[:100]}...")
        print(f"      Razon:    {r.reasoning}")
        print()

## 4. Evolucion del benchmark

El benchmark ha mejorado progresivamente a lo largo de los sprints:

| Experimento | Sprint | Accuracy | Mejora clave |
|-------------|--------|----------|--------------|
| Exp 0 | 2.5 | 88% | Baseline inflado (25 preguntas faciles) |
| Exp 1 | 2.6 | 60% | Baseline honesto (50 preguntas, 5 resource types) |
| Exp 2 | 3.1 | 82% | Pagination fix (`_count=200`, `_summary=count`) |
| Exp 3 | 3.2 | 86% | Terminology fix, `get_resource` tool |
| Exp 4 | 3.3 | 94% | Code interpreter (sandboxed Python) |
| **Exp 5** | **3.5** | **98%** | **Judge regex fix, timeout bump** |

## 5. Ejecutar desde CLI

El benchmark tambien se puede ejecutar desde la terminal:

```bash
# Todas las preguntas
uv run python -m benchmarks.run_eval

# Solo una categoria
uv run python -m benchmarks.run_eval --category simple

# Preguntas especificas
uv run python -m benchmarks.run_eval --question S01 --question M01

# Resultados en directorio custom
uv run python -m benchmarks.run_eval --output-dir benchmarks/results
```

Los resultados se guardan como JSON en `benchmarks/results/` para analisis posterior.